# FIN1 — Where Berkeley's Money Comes From and Where It Goes

**Berkeley City Finance Curriculum · Module 1**

This notebook teaches you to read a city's money the way an auditor would:
from the city's own filed documents, deriving every number yourself.

**The one rule of this curriculum:** no number in any chart is typed in by
hand. Every figure is computed, at run time, from a *facts file* that records
where each number came from (document, page, URL). If you don't believe a
number, the provenance is right there — go check it.

**What you'll be able to answer after this module:**
1. How much money does Berkeley take in, and from whom?
2. How much does it spend, and on what?
3. Why are those two numbers different — and why is that not a scandal?
4. What are pensions, and why does every budget conversation end up there?


## 1. The facts file

Cities publish their finances in two very different documents:

- The **ACFR** (Annual Comprehensive Financial Report) — *what actually
  happened* last fiscal year, audited.
- The **Adopted Budget** — *what the Council plans* to spend next year.

We extracted the key figures from Berkeley's FY2025 ACFR and the adopted
FY2027 budget into one JSON file, with a `sources` block naming every
document. ▶ Load it and look at the provenance first — always.

In [1]:
import json, pandas as pd
from pathlib import Path

FNAME = "berkeley_budget_external_facts_2026-08-15.json"
REPO = Path.cwd()
while not (REPO / "data" / "reference" / FNAME).exists() and REPO != REPO.parent:
    REPO = REPO.parent

local = REPO / "data" / "reference" / FNAME
if local.exists():
    FACTS = json.load(open(local))
else:
    # Google Colab / standalone: fetch the facts file from the public repo
    import urllib.request
    URL = ("https://raw.githubusercontent.com/blockXblock/"
           "berkeley-housing-analysis/main/data/reference/" + FNAME)
    FACTS = json.loads(urllib.request.urlopen(URL).read())
    print("(facts file fetched from GitHub — Colab/standalone mode)")

print("facts as of:", FACTS["as_of"])
for k, v in FACTS["sources"].items():
    print(f"  {k}: {v[:95]}")

facts as of: 2026-08-15
  ACFR: Annual Comprehensive Financial Report FY2025 (year ended 6/30/2025), https://berkeleyca.gov/sit
  ADOPT: 2026-06-23 Council Special Item 01, Adoption of FY 2027 and 2028 Biennial Budget + FY 2027-2031
  PRES: 2026-05-14 Budget & Finance Policy Committee, Proposed FY 2027 & FY 2028 Biennial Budget presen
  BOOK: FY 2025 & 2026 Adopted Biennial Budget book, https://berkeleyca.gov/sites/default/files/documen
  JAN22: 2026-01-22 BFPC FY 2027 and FY 2028 Budget Update slides, https://berkeleyca.gov/sites/default/


## 2. Where the money comes from (FY2025, actual, audited)

📝 **What to read from this:** the *shape*. A city is not funded by one tax —
it is a bundle of revenue streams with different rules. Watch for three
groups: taxes on property **value**, taxes on building **size** (parcel
taxes), and taxes on what you **buy**. That VALUE / SIZE / BUY split is the
framework for Module 2.

In [2]:
rev = {k: v for k, v in FACTS["revenues_fy2025_actual"].items() if not k.startswith("_") and k != "total"}
rev_total = FACTS["revenues_fy2025_actual"]["total"]

df = (pd.Series(rev, name="fy2025_actual_$").sort_values(ascending=False).to_frame())
df["share_%"] = (df["fy2025_actual_$"] / rev_total * 100).round(1)
residual = rev_total - df["fy2025_actual_$"].sum()
print(f"Citywide (all-funds) revenue, FY2025 actual: ${rev_total:,.0f}")
print(f"extraction residual (ACFR total minus the itemized lines): ${residual:,.0f} "
      f"({residual/rev_total*100:.2f}%) — small lines not itemized in the facts file; we carry it explicitly")
df.style.format({"fy2025_actual_$": "${:,.0f}"})

Citywide (all-funds) revenue, FY2025 actual: $619,580,456
extraction residual (ACFR total minus the itemized lines): $849,463 (0.14%) — small lines not itemized in the facts file; we carry it explicitly


,fy2025_actual_$,share_%
charges_for_services,"$175,162,915",28.300000
property_tax_general,"$148,363,839",23.900000
grants_operating,"$44,337,052",7.200000
investment_earnings,"$41,326,795",6.700000
business_license_tax,"$31,576,067",5.100000
other_taxes,"$31,479,208",5.100000
parcel_tax_library,"$26,466,731",4.300000
sales_tax,"$19,962,225",3.200000
utility_users_tax,"$19,446,965",3.100000
parcel_tax_parks,"$18,689,255",3.000000


In [3]:
# Group the streams for the flow diagram — the grouping is OURS (a lens),
# the dollars are the ACFR's.
GROUPS = {
    "Property tax (VALUE)": ["property_tax_general", "property_tax_debt_service"],
    "Parcel taxes (SIZE)": ["parcel_tax_library", "parcel_tax_parks", "parcel_tax_fire", "parcel_tax_paramedic"],
    "Sales & use taxes (BUY)": ["sales_tax"],
    "Other taxes": ["utility_users_tax", "business_license_tax", "transient_occupancy_tax", "other_taxes"],
    "Charges for services": ["charges_for_services"],
    "Grants & subventions": ["grants_operating", "grants_capital", "state_subventions"],
    "Investment & misc": ["investment_earnings", "miscellaneous"],
}
grouped = {g: sum(rev[k] for k in keys) for g, keys in GROUPS.items()}
grouped["Unattributed (extraction residual)"] = residual
assert abs(sum(grouped.values()) - rev_total) < 1, "grouping + residual must conserve the total"
for g, v in sorted(grouped.items(), key=lambda x: -x[1]):
    print(f"  {g:28s} ${v/1e6:7.1f}M  ({v/rev_total*100:4.1f}%)")

  Charges for services         $  175.2M  (28.3%)
  Property tax (VALUE)         $  166.5M  (26.9%)
  Other taxes                  $   90.4M  (14.6%)
  Parcel taxes (SIZE)          $   65.4M  (10.6%)
  Grants & subventions         $   58.0M  ( 9.4%)
  Investment & misc            $   43.4M  ( 7.0%)
  Sales & use taxes (BUY)      $   20.0M  ( 3.2%)
  Unattributed (extraction residual) $    0.8M  ( 0.1%)


### ▶ The revenue flow

📝 *Before you look:* a Sankey diagram makes flows proportional — a ribbon
twice as wide is twice as much money. Find the widest ribbon. It is **not**
property tax.

In [4]:
import plotly.graph_objects as go

labels = list(grouped.keys()) + [f"All city revenues FY2025 (${rev_total/1e6:,.0f}M)"]
target = len(grouped)
fig = go.Figure(go.Sankey(
    node=dict(label=labels, pad=18, thickness=16),
    link=dict(
        source=list(range(len(grouped))),
        target=[target] * len(grouped),
        value=[grouped[g] for g in grouped],
    ),
))
fig.update_layout(title="Berkeley citywide revenues, FY2025 actual (ACFR) — derived, not typed",
                  height=430, margin=dict(l=10, r=10, t=40, b=10))
fig.show()

📝 **What this could mislead you about:**
- *Charges for services* is the biggest single stream — but much of it is
  enterprise activity (refuse, marina, permits) where the charge funds the
  service that collects it. It is not free money the Council can move around.
- The diagram shows one year. Grants and investment earnings swing hard
  year to year; parcel taxes are stable by design.
- This is **citywide, all funds** — not the General Fund. The General Fund
  (the money the Council actually steers) is a subset.

## 3. Where the money goes (FY2027, adopted plan)

📝 The spending side comes from a different document (the adopted budget)
for a different year (FY2027) — so the totals will NOT match the revenue
chart, and *that mismatch is the first exercise in honest reading*
(section 4).

In [5]:
spend = FACTS["spend_fy2027_adopted"]
spend_total = spend["total"]
dept = pd.Series(spend["by_department"]).sort_values(ascending=False)
assert abs(dept.sum() - spend_total) <= 2, "departments must sum to the adopted total (±$2 rounding)"

TOP_N = 8
top = dept.head(TOP_N)
other = dept.iloc[TOP_N:].sum()
flows = list(top.items()) + [(f"All other ({len(dept)-TOP_N} depts)", other)]

labels = [f"FY2027 adopted budget (${spend_total/1e6:,.0f}M)"] + [n for n, _ in flows]
fig = go.Figure(go.Sankey(
    node=dict(label=labels, pad=18, thickness=16),
    link=dict(source=[0]*len(flows), target=list(range(1, len(flows)+1)),
              value=[v for _, v in flows]),
))
fig.update_layout(title="Berkeley adopted spending by department, FY2027 — derived, not typed",
                  height=460, margin=dict(l=10, r=10, t=40, b=10))
fig.show()

📝 **What to notice:** Public Works and Health, Housing & Community
Services outrank Police — most people guess wrong. And the **category** view
below tells you *why* budgets are hard to cut: salaries and benefits are the
dominant category, and they are contractual.

In [6]:
cat = pd.Series(spend["by_category"]).sort_values(ascending=False)
assert abs(cat.sum() - spend_total) <= 2
for k, v in cat.items():
    print(f"  {k:28s} ${v/1e6:7.1f}M  ({v/spend_total*100:4.1f}%)")

  salaries_benefits            $  411.5M  (45.5%)
  internal_services_all_others $  231.8M  (25.6%)
  services_materials           $  172.2M  (19.0%)
  capital_outlay               $   89.8M  ( 9.9%)


## 4. Why revenues ≠ spending (and why that's not a scandal)

The revenue chart says one number; the spending chart says a much bigger
one. ▶ Compute the gap, then read the four honest reasons.

In [7]:
gap = spend_total - rev_total
print(f"FY2027 adopted spend  ${spend_total/1e6:,.1f}M")
print(f"FY2025 actual revenue ${rev_total/1e6:,.1f}M")
print(f"difference            ${gap/1e6:,.1f}M")

FY2027 adopted spend  $905.2M
FY2025 actual revenue $619.6M
difference            $285.6M


Four reasons, all structural:
1. **Different years** — FY2025 actuals vs FY2027 plan (two years of growth).
2. **Internal services double-count** — departments "buy" IT, fleet, and
   building services from each other; the adopted all-funds total counts the
   dollar on both sides.
3. **Capital spending** draws on fund balances and bond proceeds — money
   raised in *earlier* years.
4. **Adopted ≠ actual** — budgets authorize; actuals under-run.

**The skill:** when someone quotes "Berkeley's billion-dollar budget" or
"Berkeley only takes in $620M," they are both quoting real documents — for
different questions. Always ask: *which document, which year, which funds?*

## 5. The pension undertow

Every California city-budget conversation arrives here. ▶ Derive Berkeley's
employer pension contributions and rates from the ACFR figures.

In [8]:
p = FACTS["pensions"]
c = p["calpers_fy2025_actual"]
rates = p["employer_rates_pct_of_payroll"]
sal = FACTS["spend_fy2027_adopted"]["by_category"]["salaries_benefits"]
print(f"CalPERS employer contributions, FY2025 actual: ${c['total']/1e6:.1f}M")
for k in ("miscellaneous", "fire", "police"):
    print(f"  {k:14s} ${c[k]/1e6:5.1f}M   employer rate {rates[k]:.2f}% of payroll")
print(f"Net pension liability (6/30/2025): ${p['net_pension_liability_2025_06_30']/1e6:,.0f}M")
print(f"\nFor scale: total FY2027 salaries+benefits budget = ${sal/1e6:,.0f}M")
print(f"CalPERS contribution ≈ {c['total']/sal*100:.0f}% of that category")

CalPERS employer contributions, FY2025 actual: $77.9M
  miscellaneous  $ 41.9M   employer rate 38.23% of payroll
  fire           $ 13.4M   employer rate 57.21% of payroll
  police         $ 22.6M   employer rate 87.04% of payroll
Net pension liability (6/30/2025): $686M

For scale: total FY2027 salaries+benefits budget = $411M
CalPERS contribution ≈ 19% of that category


📝 **Read the rates, not just the totals.** For every $100 of police
payroll, the city sends CalPERS ~$87 *on top*. That ratio — not any single
year's deficit — is why "just hire more" is never a free choice, and it is
the deep background to the sales-tax measure on the 2026 ballot.

## 6. Integrity checks

The curriculum's discipline in miniature: the notebook re-verifies that its
own derivations conserve the source totals. If the facts file changes, these
cells — not a human's memory — catch it.

In [9]:
checks = {
    "itemized revenues + residual = ACFR total": abs(sum(rev.values()) + residual - rev_total) < 1,
    "revenue residual is small (<0.5%)": abs(residual) / rev_total < 0.005,
    "revenue groups conserve the total": abs(sum(grouped.values()) - rev_total) < 1,
    "departments sum to adopted total (±$2)": abs(dept.sum() - spend_total) <= 2,
    "categories sum to adopted total (±$2)": abs(cat.sum() - spend_total) <= 2,
    "CalPERS components sum to its total": abs(sum(c[k] for k in ('miscellaneous','fire','police')) - c['total']) < 1,
}
for name, ok in checks.items():
    print(("✅" if ok else "❌"), name)
assert all(checks.values()), "integrity check failed — inspect the facts file"
print("\nALL CHECKS PASS")

✅ itemized revenues + residual = ACFR total
✅ revenue residual is small (<0.5%)
✅ revenue groups conserve the total
✅ departments sum to adopted total (±$2)
✅ categories sum to adopted total (±$2)
✅ CalPERS components sum to its total

ALL CHECKS PASS


## Exercises

1. Which revenue stream would fall fastest in a recession? Which would barely
   move? (Hint: VALUE vs SIZE vs BUY — Prop 13 makes one of them very slow.)
2. The four parcel taxes together raise more than the sales tax. Compute the
   ratio from `grouped`. What does that tell you about where Berkeley's
   voters have historically said yes?
3. `Non-Departmental` is a top-five "department." It has no employees. Look
   at the adopted budget (source `ADOPT` in the facts file) and find out what
   lives there.
4. Recompute the spending Sankey with `TOP_N = 12`. Which departments appear
   that you had never heard of?
5. **Harder:** the FY2027 General Fund pension budget is in the facts file
   (`gf_pension_budget_fy2027`). Compare it to the FY2025 actual citywide
   contribution. What are the two reasons the numbers differ?

## Where this goes next

- **FIN2 — The three ways a city taxes you** (VALUE / SIZE / BUY, and who
  actually pays, parcel by parcel)
- **FIN3 — How to read a bond** (Measure U's Tax Rate Statement, taken apart)
- **FIN4 — How a project list becomes a ballot measure** (the King Pool
  paper trail: commission → survey → silent re-scope)
- **FIN5 — The structural deficit** (why costs grow faster than revenues)
- **FIN6 — Ballot measures and the General Fund** (what each 2026 measure
  does to the money you just mapped)

*Sources: see the `sources` block printed in section 1 — every figure above
traces to a page of the ACFR FY2025 or the adopted FY2027–28 budget.*